In [ ]:
# @title ##### License { display-mode: "form" }
# Copyright 2019 DeepMind Technologies Ltd. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# OpenSpiel

* This Colab gets you started with installing OpenSpiel and its dependencies.
* OpenSpiel is a framework for reinforcement learning in games.
* The instructions are adapted from [here](https://github.com/deepmind/open_spiel/blob/master/docs/install.md).

## Install

Install OpenSpiel via pip:


In [3]:
%pip install --upgrade open_spiel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.3 MB/s eta 0:00:00


# It's play time!

In [2]:
import numpy as np
import pyspiel

game = pyspiel.load_game("tic_tac_toe")
state = game.new_initial_state()

while not state.is_terminal():
  state.apply_action(np.random.choice(state.legal_actions()))
  print(str(state) + "\n")

...
...
x..

.o.
...
x..

.ox
...
x..

.ox
.o.
x..

xox
.o.
x..

xox
.oo
x..

xox
.oo
x.x

xox
.oo
xox



In [4]:
import pyspiel
from open_spiel.python.algorithms import cfr
from open_spiel.python.algorithms import exploitability

# 1. Load the Leduc Poker game
game = pyspiel.load_game("leduc_poker")

# 2. Initialize the CFR solver
cfr_solver = cfr.CFRSolver(game)

# 3. Train the algorithm and track exploitability over iterations
num_iterations = 1000

for i in range(num_iterations):
    cfr_solver.evaluate_and_update_policy()

    # Print progress every 100 iterations
    if (i + 1) % 100 == 0:
        conv = exploitability.exploitability(game, cfr_solver.average_policy())
        print(f"Iteration {i + 1}/{num_iterations} - Exploitability: {conv:.6f}")

# 4. Extract the trained average policy
average_policy = cfr_solver.average_policy()

print("\nTraining complete! Average policy is ready for evaluation.")

Iteration 100/1000 - Exploitability: 0.095716
Iteration 200/1000 - Exploitability: 0.053838
Iteration 300/1000 - Exploitability: 0.035524
Iteration 400/1000 - Exploitability: 0.026436
Iteration 500/1000 - Exploitability: 0.021507
Iteration 600/1000 - Exploitability: 0.019776
Iteration 700/1000 - Exploitability: 0.016319
Iteration 800/1000 - Exploitability: 0.014362
Iteration 900/1000 - Exploitability: 0.014060
Iteration 1000/1000 - Exploitability: 0.011818

Training complete! Average policy is ready for evaluation.


In [7]:
import random
import pyspiel
from open_spiel.python.algorithms import cfr

# Mappings for human readability
CARD_NAMES = {
    0: "Jack ♠", 1: "Jack ♥",
    2: "Queen ♠", 3: "Queen ♥",
    4: "King ♠", 5: "King ♥"
}
ACTION_NAMES = {0: "Fold", 1: "Call / Check", 2: "Raise"}

# 1. Load Game and Train CFR Solver
game = pyspiel.load_game("leduc_poker")
cfr_solver = cfr.CFRSolver(game)

print("Training CFR algorithm over 2,000 iterations...")
for _ in range(2000):
    cfr_solver.evaluate_and_update_policy()

# Extract the converged average policy
trained_policy = cfr_solver.average_policy()
print("Training complete! Playing a game using learned equilibrium strategies.\n")


# 2. Function to Play a Game Using the Trained Policy
def play_trained_game(state, player_cards=None, public_card=None):
    if player_cards is None:
        player_cards = {}

    # Terminal State
    if state.is_terminal():
        returns = state.returns()
        print(f"🏁 GAME OVER | Payoffs -> Player 0: {returns[0]:+.1f}, Player 1: {returns[1]:+.1f}\n")
        return

    # Chance Node (Card Dealing)
    if state.is_chance_node():
        outcomes = state.chance_outcomes()
        actions, probs = zip(*outcomes)

        # Sample card dealing according to true probability
        chosen_action = random.choices(actions, weights=probs)[0]
        card_label = CARD_NAMES[chosen_action]

        if len(player_cards) == 0:
            player_cards[0] = card_label
            print(f"🎲 CHANCE: Dealt {card_label} to Player 0")
        elif len(player_cards) == 1:
            player_cards[1] = card_label
            print(f"🎲 CHANCE: Dealt {card_label} to Player 1")
        else:
            public_card = card_label
            print(f"🎲 CHANCE: Flop revealed {card_label}")

        state.apply_action(chosen_action)
        play_trained_game(state, player_cards, public_card)
        return

    # Decision Node (Player Turn using CFR Policy)
    curr_player = state.current_player()
    info_state_str = state.information_state_string(curr_player)

    # Get action probabilities recommended by the trained CFR policy
    state_policy = trained_policy.action_probabilities(state)

    actions = list(state_policy.keys())
    probabilities = list(state_policy.values())

    # Display the learned strategy breakdown for this turn
    strat_display = ", ".join([f"{ACTION_NAMES[a]}: {p:.1%}" for a, p in state_policy.items()])
    print(f"👤 Player {curr_player} (Holding: {player_cards[curr_player]} | Flop: {public_card or 'None'})")
    print(f"   Learned Strategy Probabilities -> [{strat_display}]")

    # Sample an action weighted by the CFR probabilities
    chosen_action = random.choices(actions, weights=probabilities)[0]
    print(f"   -> Executed Action: {ACTION_NAMES[chosen_action]}\n")

    state.apply_action(chosen_action)
    play_trained_game(state, player_cards, public_card)


# Run a simulation using trained decisions
play_trained_game(game.new_initial_state())

Training CFR algorithm over 2,000 iterations...
Training complete! Playing a game using learned equilibrium strategies.

🎲 CHANCE: Dealt Jack ♠ to Player 0
🎲 CHANCE: Dealt King ♠ to Player 1
👤 Player 0 (Holding: Jack ♠ | Flop: None)
   Learned Strategy Probabilities -> [Call / Check: 92.1%, Raise: 7.9%]
   -> Executed Action: Call / Check

👤 Player 1 (Holding: King ♠ | Flop: None)
   Learned Strategy Probabilities -> [Call / Check: 0.2%, Raise: 99.8%]
   -> Executed Action: Raise

👤 Player 0 (Holding: Jack ♠ | Flop: None)
   Learned Strategy Probabilities -> [Fold: 94.9%, Call / Check: 3.5%, Raise: 1.6%]
   -> Executed Action: Fold

🏁 GAME OVER | Payoffs -> Player 0: -1.0, Player 1: +1.0



In [5]:
import random
import pyspiel
from open_spiel.python.algorithms import cfr

# Mappings for human readability
CARD_NAMES = {
    0: "Jack ♠", 1: "Jack ♥", 2: "Jack ♦",
    3: "Queen ♠", 4: "Queen ♥", 5: "Queen ♦",
    6: "King ♠", 7: "King ♥", 8: "King ♦"
}
ACTION_NAMES = {0: "Fold", 1: "Call / Check", 2: "Raise"}

# 1. Load Game and Train
game = pyspiel.load_game("leduc_poker(players=3)")
cfr_solver = cfr.CFRSolver(game)

print(f"Training CFR for 3 players over 1,000 iterations...")
for _ in range(1000):
    cfr_solver.evaluate_and_update_policy()

trained_policy = cfr_solver.average_policy()
print("Training complete! Running fast simulation...\n")

# 2. Fast Iterative Simulation (No recursion overhead)
state = game.new_initial_state()
player_cards = {}
public_card = None

while not state.is_terminal():
    # Chance Node (Card Dealing)
    if state.is_chance_node():
        outcomes = state.chance_outcomes()
        actions, probs = zip(*outcomes)

        chosen_action = random.choices(actions, weights=probs)[0]
        card_label = CARD_NAMES.get(chosen_action, f"Card {chosen_action}")

        if len(player_cards) < 3:
            p_idx = len(player_cards)
            player_cards[p_idx] = card_label
            print(f"🎲 CHANCE: Dealt {card_label} to Player {p_idx}")
        else:
            public_card = card_label
            print(f"🎲 CHANCE: Flop revealed {card_label}")

        state.apply_action(chosen_action)

    # Decision Node (Player Turn)
    else:
        curr_player = state.current_player()

        # Fast action lookup from policy
        state_policy = trained_policy.action_probabilities(state)
        actions = list(state_policy.keys())
        probabilities = list(state_policy.values())

        strat_display = ", ".join([f"{ACTION_NAMES[a]}: {p:.1%}" for a, p in state_policy.items()])
        print(f"👤 Player {curr_player} (Holding: {player_cards.get(curr_player, 'Unknown')} | Flop: {public_card or 'None'})")
        print(f"   Strategy -> [{strat_display}]")

        chosen_action = random.choices(actions, weights=probabilities)[0]
        print(f"   -> Executed: {ACTION_NAMES[chosen_action]}\n")

        state.apply_action(chosen_action)

# Terminal State
returns = state.returns()
payoff_str = ", ".join([f"Player {i}: {returns[i]:+.1f}" for i in range(3)])
print(f"🏁 GAME OVER | Payoffs -> {payoff_str}")

Training CFR for 3 players over 1,000 iterations...


KeyboardInterrupt: 

In [ ]:
import random
import pyspiel
from open_spiel.python.algorithms import cfr

# Mappings for human readability
CARD_NAMES = {
    0: "Jack ♠", 1: "Jack ♥", 2: "Jack ♦",
    3: "Queen ♠", 4: "Queen ♥", 5: "Queen ♦",
    6: "King ♠", 7: "King ♥", 8: "King ♦"
}
ACTION_NAMES = {0: "Fold", 1: "Call / Check", 2: "Raise"}

# 1. Load Game and Train
game = pyspiel.load_game("leduc_poker(players=3)")
cfr_solver = cfr.CFRSolver(game)

print(f"Training CFR for 3 players over 1,000 iterations...")
for _ in range(1000):
    cfr_solver.evaluate_and_update_policy()

trained_policy = cfr_solver.average_policy()
print("Training complete! Running fast simulation...\n")

# 2. Fast Iterative Simulation (No recursion overhead)
state = game.new_initial_state()
player_cards = {}
public_card = None

while not state.is_terminal():
    # Chance Node (Card Dealing)
    if state.is_chance_node():
        outcomes = state.chance_outcomes()
        actions, probs = zip(*outcomes)

        chosen_action = random.choices(actions, weights=probs)[0]
        card_label = CARD_NAMES.get(chosen_action, f"Card {chosen_action}")

        if len(player_cards) < 3:
            p_idx = len(player_cards)
            player_cards[p_idx] = card_label
            print(f"🎲 CHANCE: Dealt {card_label} to Player {p_idx}")
        else:
            public_card = card_label
            print(f"🎲 CHANCE: Flop revealed {card_label}")

        state.apply_action(chosen_action)

    # Decision Node (Player Turn)
    else:
        curr_player = state.current_player()

        # Fast action lookup from policy
        state_policy = trained_policy.action_probabilities(state)
        actions = list(state_policy.keys())
        probabilities = list(state_policy.values())

        strat_display = ", ".join([f"{ACTION_NAMES[a]}: {p:.1%}" for a, p in state_policy.items()])
        print(f"👤 Player {curr_player} (Holding: {player_cards.get(curr_player, 'Unknown')} | Flop: {public_card or 'None'})")
        print(f"   Strategy -> [{strat_display}]")

        chosen_action = random.choices(actions, weights=probabilities)[0]
        print(f"   -> Executed: {ACTION_NAMES[chosen_action]}\n")

        state.apply_action(chosen_action)

# Terminal State
returns = state.returns()
payoff_str = ", ".join([f"Player {i}: {returns[i]:+.1f}" for i in range(3)])
print(f"🏁 GAME OVER | Payoffs -> {payoff_str}")

In [2]:
import random
import pyspiel
from open_spiel.python.algorithms import external_sampling_mccfr

# Mappings for human readability
CARD_NAMES = {
    0: "Jack ♠", 1: "Jack ♥", 2: "Jack ♦",
    3: "Queen ♠", 4: "Queen ♥", 5: "Queen ♦",
    6: "King ♠", 7: "King ♥", 8: "King ♦"
}
ACTION_NAMES = {0: "Fold", 1: "Call / Check", 2: "Raise"}

# 1. Load 3-Player Leduc Poker
game = pyspiel.load_game("leduc_poker(players=3)")

# 2. Initialize Fast MCCFR Solver
# Full sampling ensures we compute a true average policy across iterations
mccfr_solver = external_sampling_mccfr.ExternalSamplingSolver(
    game,
    external_sampling_mccfr.AverageType.FULL
)

print("Training 3-player MCCFR over 5,000 iterations (takes ~2-3 seconds)...")
for i in range(1, 300):
    mccfr_solver.iteration()
    if i % 1000 == 0:
        print(f"  -> Completed {i}/5000 iterations")

trained_policy = mccfr_solver.average_policy()
print("\nTraining complete! Running step-by-step game simulation...\n")

# 3. Iterative Playout (No recursion overhead)
state = game.new_initial_state()
player_cards = {}
public_card = None

while not state.is_terminal():
    # Chance Node (Card Dealing)
    if state.is_chance_node():
        outcomes = state.chance_outcomes()
        actions, probs = zip(*outcomes)

        chosen_action = random.choices(actions, weights=probs)[0]
        card_label = CARD_NAMES.get(chosen_action, f"Card {chosen_action}")

        if len(player_cards) < 3:
            p_idx = len(player_cards)
            player_cards[p_idx] = card_label
            print(f"🎲 CHANCE: Dealt {card_label} to Player {p_idx}")
        else:
            public_card = card_label
            print(f"🎲 CHANCE: Flop revealed {card_label}")

        state.apply_action(chosen_action)

    # Decision Node (Player Turn)
    else:
        curr_player = state.current_player()

        # Extract action probabilities recommended by the learned policy
        state_policy = trained_policy.action_probabilities(state)
        actions = list(state_policy.keys())
        probabilities = list(state_policy.values())

        strat_display = ", ".join([f"{ACTION_NAMES[a]}: {p:.1%}" for a, p in state_policy.items()])
        print(f"👤 Player {curr_player} (Holding: {player_cards.get(curr_player, 'Unknown')} | Flop: {public_card or 'None'})")
        print(f"   Learned Strategy Probabilities -> [{strat_display}]")

        # Pick action according to trained probabilities
        chosen_action = random.choices(actions, weights=probabilities)[0]
        print(f"   -> Executed Action: {ACTION_NAMES[chosen_action]}\n")

        state.apply_action(chosen_action)

# Terminal State
returns = state.returns()
payoff_str = ", ".join([f"Player {i}: {returns[i]:+.1f}" for i in range(3)])
print(f"🏁 GAME OVER | Payoffs -> {payoff_str}")

Training 3-player MCCFR over 5,000 iterations (takes ~2-3 seconds)...

Training complete! Running step-by-step game simulation...

🎲 CHANCE: Dealt Queen ♠ to Player 0
🎲 CHANCE: Dealt Queen ♦ to Player 1
🎲 CHANCE: Dealt Jack ♥ to Player 2
👤 Player 0 (Holding: Queen ♠ | Flop: None)
   Learned Strategy Probabilities -> [Call / Check: 97.7%, Raise: 2.3%]
   -> Executed Action: Call / Check

👤 Player 1 (Holding: Queen ♦ | Flop: None)
   Learned Strategy Probabilities -> [Call / Check: 97.5%, Raise: 2.5%]
   -> Executed Action: Call / Check

👤 Player 2 (Holding: Jack ♥ | Flop: None)
   Learned Strategy Probabilities -> [Call / Check: 74.5%, Raise: 25.5%]
   -> Executed Action: Call / Check

🎲 CHANCE: Flop revealed Jack ♦
👤 Player 0 (Holding: Queen ♠ | Flop: Jack ♦)
   Learned Strategy Probabilities -> [Call / Check: 50.0%, Raise: 50.0%]
   -> Executed Action: Call / Check

👤 Player 1 (Holding: Queen ♦ | Flop: Jack ♦)
   Learned Strategy Probabilities -> [Call / Check: 47.1%, Raise: 52.9%]
  